# Generated queries and their judgements

A read-only view of one generation round: what the operators minted, and what the
coherence judge said about it. No LLM calls, no Qdrant, nothing written.

The gate asks two things of every row it holds back: could a real user type this
query, and does the document paired with it actually answer it. A floor's credit
gate opens only once its whole staged pilot is judged AND passes at
`config.coherence_pass_rate`, so a single unjudged pilot row keeps a floor clamped.

| artifact | what it holds |
|---|---|
| `data/augmentation/pool.parquet` | every generated row: operator, floor, credit gate, parent |
| `data/augmentation/coherence_audit.parquet` | one verdict per judged row, with the model's reason |
| `data/augmentation/constructed.parquet` | answer docs minted for the synthetic rung |
| `data/v3/dataset_v3.parquet` | the composed dataset, to see what actually landed |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from augmentation.config import AugmentationConfig
from augmentation.constructed import ConstructedDocs
from augmentation.judge import CoherenceJudge
from augmentation.pool import GeneratedPool

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

config = AugmentationConfig()
pool = GeneratedPool().load()
judge = CoherenceJudge(config=config)
verdicts = judge.load()
constructed = ConstructedDocs().load()

print(f"pool       : {len(pool):,} generated rows, {pool['operator'].nunique()} operators")
print(f"verdicts   : {len(verdicts):,} judged rows"
      + (f", model {verdicts['model'].iloc[0]}" if len(verdicts) else ""))
print(f"constructed: {len(constructed):,} answer docs")
print(f"dials      : coherence_pass_rate={config.coherence_pass_rate} "
      f"pilot_n={config.pilot_n} judge_model={config.engine.model}")

## Coverage: how much of the gate has an answer at all

`passed()` is the set admission reads. Everything unjudged is neither passed nor
failed, it is simply held, and one LLM call each is what it costs to find out.

In [ ]:
gated = CoherenceJudge.candidates(pool)
is_judged = gated["query_id"].astype(str).isin(set(verdicts["query_id"].astype(str)))

print(f"coherence-gated rows  : {len(gated):,}")
print(f"  judged              : {int(is_judged.sum()):,}")
print(f"  unjudged            : {int((~is_judged).sum()):,}")
print(f"passed() -> admissible: {len(judge.passed()):,}")

pool["credit_gate"].fillna("none").value_counts().to_frame("rows")

## Verdict split, and which floors clear the dial

`opens_gate` is the whole point of the judge in stage 3: a floor that clears it
gets its full row demand, every other gated floor stays clamped to `pilot_n`.

In [ ]:
print(verdicts["verdict"].value_counts().rename({True: "pass", False: "fail"}).to_string())
print(f"\noverall pass rate: {verdicts['verdict'].mean():.1%}")

by_floor = (
    verdicts.groupby("floor")["verdict"]
    .agg(judged="size", pass_rate="mean")
    .sort_values("pass_rate", ascending=False)
)
by_floor["opens_gate"] = by_floor["pass_rate"] >= config.coherence_pass_rate
print(f"\n{int(by_floor['opens_gate'].sum())} of {len(by_floor)} judged floors clear "
      f"{config.coherence_pass_rate}")
by_floor.round(3)

## Sample judgements

`evidence` is resolved through the judge's own `_evidence`, so what you read here
is the text the verdict actually saw rather than a reconstruction of it. Rows
grounded in a lane document need the parent pool to resolve, which is why it is
built here.

In [ ]:
from augmentation.parents import ParentPool
from composition.composer import V3Composition
from composition.floors import read_catalog
from dataset_registry import DATASETS

REGISTRY = {d.name: d for d in DATASETS}
composer = V3Composition()
parents = ParentPool(
    read_catalog(composer.catalog_path),
    pd.read_parquet(composer.dataset_path).astype({"query_id": str}),
    REGISTRY,
)
reader = CoherenceJudge(config=config, parents=parents)
judged_pool = pool.merge(verdicts, on="query_id", suffixes=("", "_v"))


def judgements(n=6, verdict=None, floor=None, operator=None, seed=0, chars=400):
    """Sampled judged rows next to the evidence their verdict read."""
    rows = judged_pool
    if verdict is not None:
        rows = rows[rows["verdict"] == verdict]
    if floor is not None:
        rows = rows[rows["floor"] == floor]
    if operator is not None:
        rows = rows[rows["operator"] == operator]
    if rows.empty:
        return rows
    sample = rows.sample(n=min(n, len(rows)), random_state=seed)
    return pd.DataFrame([
        {
            "floor": row.floor,
            "operator": row.operator,
            "verdict": "pass" if row.verdict else "FAIL",
            "reason": row.reason,
            "query": row.query,
            "evidence": reader._evidence(row, constructed)[:chars],
        }
        for row in sample.itertuples(index=False)
    ])


judgements(n=6, verdict=False)

Now the passes, for the same read. Change `floor=` or `operator=` to drill into
one archetype, and raise `n` freely: the sampling is seeded, so a rerun is stable.

In [ ]:
judgements(n=6, verdict=True)

## Why rows fail

The reason is one clause of at most twelve words, so exact repeats are meaningful:
a phrasing that recurs is the judge hitting the same structural problem.

In [ ]:
fails = verdicts[~verdicts["verdict"].astype(bool)]
print(f"{len(fails):,} failed verdicts, {fails['reason'].nunique():,} distinct reasons\n")
print(fails["reason"].value_counts().head(15).to_string())

words = (
    fails["reason"].str.lower().str.findall(r"[a-z]{4,}")
    .explode().value_counts().head(20)
)
words.to_frame("mentions")

## Generated queries by operator

Ignoring the gate entirely: what each operator produces. `lane_synthesize` is the
class-residual carrier and dominates the pool by volume.

In [ ]:
print(pool["operator"].value_counts().to_string())

pd.concat([
    group.sample(n=min(3, len(group)), random_state=0)[["operator", "floor", "query"]]
    for _, group in pool.groupby("operator")
], ignore_index=True)

## Rewrites next to their parents

Only operators that rewrite an existing query have a parent. `lane_synthesize` and
`synthesize` mint from scratch and carry an empty `generated_from`, so they are
excluded here. `meaning_preserved` is the operator's own declaration about whether
the rewrite was supposed to keep the query's meaning.

In [ ]:
from composition.compose import join_text

rewrites = pool[
    (pool["generated_from"].astype(str) != "")
    & pool["parent_dataset"].isin(REGISTRY)
]
sample = rewrites.sample(n=min(8, len(rewrites)), random_state=0)
keys = sample.assign(
    dataset=sample["parent_dataset"],
    query_id=sample["generated_from"].astype(str),
)
sample = sample.assign(parent_query=join_text(keys, REGISTRY).to_numpy())
sample[["operator", "floor", "meaning_preserved", "parent_query", "query"]]

## What actually reached the dataset, by rung

`provenance` separates the rungs: `natural` never went through generation,
`augmented` is a rewrite of a real query, `doc_grounded` was minted against a real
lane document, `synthetic` was minted from scratch. A generated row only enters the
dataset once it is LABELLED, so a rung with rows in the pool and none here has an
open edge rather than a quality problem.

In [ ]:
dataset = pd.read_parquet(composer.dataset_path).astype({"query_id": str})

print(f"dataset_v3: {len(dataset):,} rows")
print()
print(dataset["provenance"].value_counts(dropna=False).to_frame("rows").to_string())

landed = dataset.merge(
    pool[["query_id", "operator", "floor", "provenance"]],
    on="query_id", suffixes=("", "_pool"),
)
print()
print(f"{len(landed):,} match a generated-pool row. Which rung minted them:")
print()
if len(landed):
    print(pd.crosstab(landed["operator"], landed["route_class"]).to_string())
    print()
    print(f"certified: {int(landed['certified'].sum())} of {len(landed)}")

### The funnel, per operator

Where each rung's rows currently stand. The two rings close differently: cell and
corruption floors run generate -> admit -> label, while the lane rung takes its
demand from `lane_order.parquet`, never passes through `admit`, and is labelled
straight from the judge's passed set. So `floor_on_sheet` is expected to be 0 for
`lane_synthesize`, and `admitted` is not a step on its path.

In [ ]:
sheet_floors = set(pd.read_parquet(composer.order_sheet_path)["floor"])
in_dataset = set(dataset["query_id"])
admitted_ids = (
    set(pd.read_parquet(composer.admitted_path)["query_id"].astype(str))
    if composer.admitted_path.exists() else set()
)
cleared = judge.passed()
ids = pool["query_id"].astype(str)


def funnel(group: pd.DataFrame) -> pd.Series:
    gid = group["query_id"].astype(str)
    gate = group["credit_gate"].fillna("none")
    return pd.Series({
        "minted": len(group),
        "floor_on_sheet": int(group["floor"].isin(sheet_floors).sum()),
        "gate_open": int((gate.eq("none") | gid.isin(cleared)).sum()),
        "admitted": int(gid.isin(admitted_ids).sum()),
        "in_dataset": int(gid.isin(in_dataset).sum()),
    })


pool.groupby("operator").apply(funnel, include_groups=False).sort_values(
    "minted", ascending=False
)

## The unjudged backlog

What one more judge pass would cost and buy, at the pass rate measured above.
Run it with `poetry run python src/scripts/run_v3_generation.py --llm-coherence`,
whose stage 5b is idempotent on `query_id`, so it only pays for rows still unjudged.

In [ ]:
unjudged = gated[~is_judged]
rate = float(verdicts["verdict"].mean()) if len(verdicts) else float("nan")
print(f"{len(unjudged):,} rows unjudged, one call each at {config.engine.model}")
print(f"expected passes at the measured rate ({rate:.1%}): ~{int(len(unjudged) * rate):,}")

unjudged.groupby(["operator", "floor"]).size().sort_values(
    ascending=False
).head(20).to_frame("unjudged")